# Distil the teacher on real receipts from Kaggle

The next SFT round, with real receipts from Kaggle added to the generated train suites. **Run `real_receipts_eval.ipynb` first:** it builds the suites, audits their gold, and scores the shipped model, the untuned 0.8B and the teacher on `dev-receipts`. This round is measured against that baseline, and its teacher row on `dev-receipts` is reused from Drive.

1. **The suites.** They come from Drive, as the evaluation notebook built them, with the same settings. If they aren't there, this notebook downloads the dataset and builds them the same way, but then nothing has been audited.
2. **The teacher on `train-receipts`, beside the generated train suites.** A train run already finished on Drive is reused.
3. **The pairs, SFT from the base model, the merge**, as in `sft_a100.ipynb`.
4. **Scores at bf16** on `dev`, `dev-novel` and `dev-receipts`, each beside the earlier rounds, the shipped model and the teacher. For a GGUF row on `dev-receipts` like the shipped one's, add the merged model's quantized file to `MODELS` in `real_receipts_eval.ipynb`.

How the receipts become cases is described in `real_receipts_eval.ipynb` and in the eval README (*Real receipts*). The receipts and the suites stay on Drive and out of git.

The GPU is the same as `sft_a100.ipynb`: **A100** while any teacher run is still to do, and an **L4** once the teacher's sessions are on Drive. A T4 doesn't work for SFT. The Kaggle key is only needed if the dataset's zip isn't on Drive yet. It is read from Colab's secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`), or typed into a hidden prompt, and passed only to the download.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

def sh(args, cwd=None, env=None):
    process = subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args[:3])


def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""


REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "claude/modest-brahmagupta-hpbdfb"
RECORDS = "eval-records"
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
sh(["rm", "-rf", CLONE])
sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE])
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

## Settings

**The dataset** (the same in `real_receipts_eval.ipynb` and `real_receipts_sft.ipynb`; the suites are built once, and a notebook whose settings differ from the suites on Drive stops). `DATASET` is the Kaggle dataset. Set `CURRENCY` for receipts that print no currency of their own: the converter recognises `RM`, `€`, `£`, `US$`, `S$`, `Rp` and other markers, and uses `CURRENCY` only where it finds none. A bare `$` names no country (Malaysian receipts print it for ringgit), so it is read as `CURRENCY` too. Categories come from keyword rules on the shop's name and the receipt's words, in English, French and Malay, and every receipt of one shop gets the same category. `MONTH_FIRST` reads `03/04/2019` as March 4th instead of April 3rd. `LANGUAGE` picks the category titles (`auto`: French if most receipts read French, otherwise English). With `OCR = "auto"`, tesseract runs only when the dataset gives no text; set `"always"` or `"never"` to force it. `OCR_LANGUAGES` lists tesseract's languages, for example `eng+fra`.

**The round.** `TEACHER_RUNS` lists the teacher runs the pairs are built from. Generated suites are regenerated and checked against their fingerprints; the real-receipt suite is marked `real` and comes from Drive. A run already finished on Drive is reused, so `teacher-train` and `teacher-train-2` cost nothing if round 4 made them. If `teacher-train-2` was never run, it costs about 2.6 hours on an A100; remove that entry to skip it. `ROUND` names the adapter, the merged model (`qwen3.5-0.8b-sft-r5`) and its runs. `PREVIOUS`, `EARLIER` and `SHIPPED` are what it is compared with. A run of them that isn't present is left out of the comparison.

In [ ]:
DATASET = "dhiaznaidi/receiptdatasetssd300v2"
CURRENCY = "MYR"
MONTH_FIRST = False
LANGUAGE = "auto"
OCR = "auto"
OCR_LANGUAGES = "eng+fra"
SEED = 51
EVAL_SHARE = 0.15
TRAIN_COPIES = 2
TRAIN_SUITE = "train-receipts"
EVAL_SUITE = "dev-receipts"
TEACHER_NAME = "qwen3.8-27b-fp8-v5"

TEACHER = "Qwen/Qwen3.8-27B-FP8"
TEACHER_RUNS = {
    "teacher-train": {"suite": "train", "fingerprint": "a5437ef135b8f48f", "cases": 2100,
                      "args": ["--per-locale", "300", "--users-per-locale", "30", "--seed", "1", "--novel-share", "0.5"]},
    "teacher-train-2": {"suite": "train-2", "fingerprint": "9c208ad58ed4d37a", "cases": 4200,
                        "args": ["--per-locale", "600", "--users-per-locale", "60", "--seed", "4", "--novel-share", "0.5"]},
    f"teacher-{TRAIN_SUITE}": {"suite": TRAIN_SUITE, "real": True},
}
STUDENT = "Qwen/Qwen3.5-0.8B"
ROUND = "r5"
PREVIOUS = "qwen3.5-0.8b-sft-r4"
EARLIER = "qwen3.5-0.8b-sft-r3"
SHIPPED = "qwen3.5-0.8b-grpo-r2"
LOSS = "token"
DOUBT_CONFIDENCE = 0.5
NAME = f"qwen3.5-0.8b-sft-{ROUND}"
EPOCHS = 1
RANK = 16
LR = 2e-4
ACCUM = 8
MAX_LEN = 8192
print(NAME)

## Drive

In [ ]:
import json
import shutil
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/spending-agent"
DATA_ZIP = f"{DRIVE}/data/{DATASET.split('/')[-1]}.zip"
RAW = f"/content/data/{DATASET.split('/')[-1]}"
SUITES_KEPT = f"{DRIVE}/suites"
for folder in (os.path.dirname(DATA_ZIP), SUITES_KEPT, f"{DRIVE}/runs"):
    os.makedirs(folder, exist_ok=True)


def keep_on_drive(run):
    kept, local = f"{DRIVE}/runs/{run}", f"{RECORDS_DIR}/runs/{run}"
    os.makedirs(kept, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local) and not os.listdir(kept):
        shutil.copytree(local, kept, dirs_exist_ok=True)
    sh(["rm", "-rf", local])
    os.symlink(kept, local)


def finished(folder, cases):
    results = f"{folder}/results.jsonl"
    return os.path.exists(results) and sum(1 for _ in open(results)) >= cases

TEACHER_DIRS = {run: f"{DRIVE}/runs/{run}" for run in TEACHER_RUNS}
TEACHER_NOVEL_RUN = f"{TEACHER_NAME}-dev-novel"
TEACHER_REAL_RUN = f"{TEACHER_NAME}-{EVAL_SUITE}"
PAIRS = f"{DRIVE}/sft/{'+'.join(TEACHER_RUNS)}" + (f"-doubt{DOUBT_CONFIDENCE:g}" if DOUBT_CONFIDENCE is not None else "") + ".jsonl"
ADAPTER = f"{DRIVE}/sft/{NAME}"
MERGED = f"{DRIVE}/models/{NAME}"
for folder in (os.path.dirname(PAIRS), ADAPTER, os.path.dirname(MERGED)):
    os.makedirs(folder, exist_ok=True)

## The receipts from Kaggle

The zip is downloaded once into `MyDrive/spending-agent/data/` and unpacked to local disk on every run. Unpacking thousands of small files onto Drive would be slow.

The key comes from Colab's secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`). Without them, the cell asks for it in a hidden prompt. It is handed only to the `kaggle` command's environment, and is never written to disk or kept on Drive.

In [ ]:
if os.path.exists(f"{SUITES_KEPT}/{TRAIN_SUITE}/meta.json") and os.path.exists(f"{SUITES_KEPT}/{EVAL_SUITE}/meta.json"):
    print("the suites are on Drive: no download needed")
else:
    print("no suites on Drive: run real_receipts_eval.ipynb first for a baseline and an audit; building them here")


import getpass

if not os.path.exists(DATA_ZIP) and not os.path.exists(f"{SUITES_KEPT}/{EVAL_SUITE}/meta.json"):
    username, key, api_token = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY"), secret("KAGGLE_API_TOKEN")
    if not api_token and not (username and key):
        print("No Kaggle secret in this notebook's Colab secrets: type it here instead.")
        typed = getpass.getpass("Kaggle API key, or a KGAT_ API token: ").strip()
        if typed.startswith("KGAT_"):
            api_token = typed
        else:
            username, key = input("Kaggle username: ").strip(), typed
    kaggle_env = {k: v for k, v in os.environ.items() if not k.startswith("KAGGLE_")}
    if api_token:
        kaggle_env["KAGGLE_API_TOKEN"] = api_token
    if username and key:
        kaggle_env.update(KAGGLE_USERNAME=username, KAGGLE_KEY=key)
    sh(["pip", "install", "-q", "-U", "kaggle"])
    download = "/content/kaggle-download"
    sh(["rm", "-rf", download])
    sh(["kaggle", "datasets", "download", DATASET, "-p", download], env=kaggle_env)
    del kaggle_env, username, key, api_token
    zips = [name for name in os.listdir(download) if name.endswith(".zip")]
    if len(zips) != 1:
        raise SystemExit(f"expected one zip from Kaggle, found {os.listdir(download)}")
    shutil.copy(f"{download}/{zips[0]}", DATA_ZIP)
    sh(["rm", "-rf", download])
if os.path.exists(DATA_ZIP) and not os.path.isdir(RAW):
    os.makedirs(RAW)
    sh(["unzip", "-q", "-o", DATA_ZIP, "-d", RAW])
print(DATA_ZIP if os.path.exists(DATA_ZIP) else "")

## What the dataset holds

File types, the largest folders, and the start of one file of each annotation type. If the *Convert* cell keeps no receipts, this output says why. Paste it into a Claude Code session to have the converter taught the layout.

In [ ]:
from collections import Counter

extensions, folders, samples = Counter(), Counter(), {}
for folder, _, names in os.walk(RAW):
    for name in names:
        ext = os.path.splitext(name)[1].lower() or "(none)"
        extensions[ext] += 1
        folders[os.path.relpath(folder, RAW)] += 1
        if ext in (".txt", ".json", ".jsonl", ".csv", ".xml", ".tsv") and ext not in samples:
            samples[ext] = os.path.join(folder, name)
print("files by type:", dict(extensions.most_common()))
print("folders:")
for folder, count in folders.most_common(15):
    print(f"  {count:6d}  {folder}")
for ext, path in samples.items():
    print(f"\n== {os.path.relpath(path, RAW)}")
    with open(path, encoding="utf-8", errors="replace") as f:
        print(f.read(1200))

## Convert

The suites are built once and kept on Drive (`MyDrive/spending-agent/suites/`); a rerun copies them back, and stops if they were built with other settings than these. The counts say how many receipts were read, by which layout (`files:`), and why the others were dropped (`dropped:`). Three cases follow, with their gold records.

**Check them.** The labels come from the dataset, and the currency, the date order and the category come from the settings and from keyword rules. If a case is wrong (a currency that is never printed, say, or US-style dates), change the setting, delete `MyDrive/spending-agent/suites/train-receipts` and `dev-receipts`, and run this cell again. Changing them after runs have started is a new suite: `run.py` refuses to resume on a different fingerprint, and the earlier runs can't be compared with the new ones.

**Text read on a Mac.** Run `ocr_macos.py` on a Mac to read the photos with macOS's own OCR, and upload its folder of `.txt` files to `MyDrive/spending-agent/data/<dataset>-macos-ocr/` (for this dataset, `receiptdatasetssd300v2-macos-ocr`). Where the dataset gives no text, the converter takes it from there instead of running tesseract here.

In [ ]:
os.chdir(EVAL)
MACOS_TEXT = f"{DRIVE}/data/{DATASET.split('/')[-1]}-macos-ocr"
TEXT_DIR = MACOS_TEXT if os.path.isdir(MACOS_TEXT) and os.listdir(MACOS_TEXT) else None
print(f"text read on a Mac: {len(os.listdir(TEXT_DIR))} files in {TEXT_DIR}" if TEXT_DIR else
      f"no text read on a Mac ({MACOS_TEXT})")
if "--text-dir" not in subprocess.run(["python3", "real_receipts.py", "--help"], capture_output=True,
                                      text=True).stdout:
    raise SystemExit(f"the real_receipts.py cloned from {BRANCH} is older than this notebook: push "
                     "eval/real_receipts.py, eval/ocr_macos.py and spending_agent/money.py, then run from the top")


def convert(ocr):
    args = ["python3", "real_receipts.py", "--source", RAW, "--dataset", DATASET, "--train-name", TRAIN_SUITE,
            "--eval-name", EVAL_SUITE, "--seed", str(SEED), "--eval-share", str(EVAL_SHARE), "--currency", CURRENCY,
            "--language", LANGUAGE, "--train-copies", str(TRAIN_COPIES), *(["--month-first"] if MONTH_FIRST else []),
            *(["--text-dir", TEXT_DIR] if TEXT_DIR else []),
            *(["--ocr", "--ocr-languages", OCR_LANGUAGES] if ocr else [])]
    try:
        sh(args)
        return True
    except subprocess.CalledProcessError:
        return False


def install_tesseract():
    languages = [l for l in OCR_LANGUAGES.split("+") if l != "eng"]
    sh(["apt-get", "-qq", "install", "-y", "tesseract-ocr", *[f"tesseract-ocr-{l}" for l in languages]])
    sh(["pip", "install", "-q", "pytesseract", "pillow"])


if all(os.path.exists(f"{SUITES_KEPT}/{name}/meta.json") for name in (TRAIN_SUITE, EVAL_SUITE)):
    for name in (TRAIN_SUITE, EVAL_SUITE):
        shutil.copytree(f"{SUITES_KEPT}/{name}", f"suites/{name}", dirs_exist_ok=True)
    print("suites from Drive")
else:
    images = sum(n for ext, n in extensions.items() if ext in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"))
    ok = OCR != "always" and convert(False)
    if ok and OCR == "auto" and images:
        with open(f"suites/{TRAIN_SUITE}/meta.json") as f:
            ok = json.load(f)["receipts"] >= 0.3 * images * (1 - EVAL_SHARE)
        if not ok:
            print("few receipts came with text: reading the images with tesseract instead")
    if not ok and OCR != "never" and images and (OCR == "always" or not TEXT_DIR):
        install_tesseract()
        ok = convert(True)
    if not ok:
        raise SystemExit("no usable receipts: see the counts above and the output of the cell before")
    for name in (TRAIN_SUITE, EVAL_SUITE):
        shutil.copytree(f"suites/{name}", f"{SUITES_KEPT}/{name}", dirs_exist_ok=True)

SUITE_CASES = {}
expected = {"dataset": DATASET, "seed": SEED, "evalShare": EVAL_SHARE, "dayFirst": not MONTH_FIRST,
            "defaultCurrency": CURRENCY, "languageSetting": LANGUAGE, "trainCopies": TRAIN_COPIES}
for name in (TRAIN_SUITE, EVAL_SUITE):
    with open(f"suites/{name}/meta.json") as f:
        meta = json.load(f)
    differs = {k: (meta.get(k), v) for k, v in expected.items() if meta.get(k) != v}
    if (OCR == "never" and meta.get("ocr")) or (OCR == "always" and not meta.get("ocr")):
        differs["ocr"] = (meta.get("ocr"), OCR)
    if bool(meta.get("textDir")) != bool(TEXT_DIR):
        differs["textDir"] = (bool(meta.get("textDir")), bool(TEXT_DIR))
    if differs:
        raise SystemExit(f"the suites on Drive were built with other settings (on Drive, here): {differs}. "
                         f"Use their settings, or delete {SUITES_KEPT}/{TRAIN_SUITE} and {SUITES_KEPT}/{EVAL_SUITE} "
                         "to build new suites (a new suite: earlier runs on them can't be compared)")
    SUITE_CASES[name] = meta["cases"]
    print(name, {k: meta[k] for k in ("cases", "users", "receipts", "fingerprint", "currencies", "language", "ocr")})
print("conversion:", json.dumps(meta["conversion"], indent=1))
with open(f"suites/{EVAL_SUITE}/cases.jsonl", encoding="utf-8") as f:
    shown = [json.loads(line) for line in f][:3]
for case in shown:
    gold = case["gold"]
    print("\n" + "=" * 80 + f"\n{case['id']}  {case['traits']}\n" + case["content"][:1500])
    print("-> gold:", {k: gold[k] for k in ("merchant", "totalAmount", "currencyCode", "purchaseDate", "purchaseTime",
                                            "category")})

## The generated train suites, and what is still to run

As in `sft_a100.ipynb`: each regenerated from its seed and checked against its fingerprint. Then each teacher run's status.

In [ ]:
os.chdir(EVAL)
for run, spec in TEACHER_RUNS.items():
    if spec.get("real"):
        spec["cases"] = SUITE_CASES[spec["suite"]]
        continue
    subprocess.run(["python3", "make_suite.py", "--name", spec["suite"], *spec["args"]], check=True, capture_output=True)
    with open(f"suites/{spec['suite']}/meta.json") as f:
        suite = json.load(f)
    print(spec["suite"], suite["cases"], "cases, fingerprint", suite["fingerprint"])
    if suite["fingerprint"] != spec["fingerprint"] or suite["cases"] != spec["cases"]:
        raise SystemExit(f"{spec['suite']} doesn't regenerate as it did: the generator changed, so this is a new suite")

for run in (*TEACHER_RUNS, TEACHER_NOVEL_RUN, TEACHER_REAL_RUN):
    keep_on_drive(run)
gpu_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.split()[0])
TEACHER_FITS = gpu_mib >= 39000
NEEDS_TRAIN_RUNS = [] if os.path.exists(PAIRS) else [
    run for run, spec in TEACHER_RUNS.items() if not finished(TEACHER_DIRS[run], spec["cases"])]
NEEDS_EVAL_RUNS = [(run, suite) for run, suite, cases in ((TEACHER_NOVEL_RUN, "dev-novel", 280),
                                                          (TEACHER_REAL_RUN, EVAL_SUITE, SUITE_CASES[EVAL_SUITE]))
                   if not finished(f"{DRIVE}/runs/{run}", cases)]
if NEEDS_TRAIN_RUNS and not TEACHER_FITS:
    raise SystemExit(f"the teacher runs {NEEDS_TRAIN_RUNS} aren't finished and this GPU has {gpu_mib} MiB: use an A100")
SERVE_TEACHER = TEACHER_FITS and (bool(NEEDS_TRAIN_RUNS) or bool(NEEDS_EVAL_RUNS))
for run in TEACHER_RUNS:
    print(f"teacher run {run}:", "to run" if run in NEEDS_TRAIN_RUNS else "done")
for run, suite in NEEDS_EVAL_RUNS:
    print(f"teacher on {suite}:", "to run" if TEACHER_FITS else "missing, and this GPU can't host the teacher")

## vLLM

As in `sft_a100.ipynb`: its own virtual environment, the teacher's `dev` flags, logs in `/content/vllm.log`.

In [ ]:
VLLM_VENV = "/content/vllm-venv"
if not os.path.exists(f"{VLLM_VENV}/bin/vllm"):
    sh(["pip", "install", "-q", "uv"])
    sh(["uv", "venv", "-q", "--python", "3.12", VLLM_VENV])
    sh(["uv", "pip", "install", "-q", "--python", f"{VLLM_VENV}/bin/python", "vllm"])
print(subprocess.run([f"{VLLM_VENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
import time
import urllib.request


def stop_servers():
    subprocess.run("pkill -f '[v]llm serve'; pkill -f '[l]lama-server'", shell=True)
    for _ in range(60):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if int(used.split()[0]) < 1000:
            return
        time.sleep(5)
    raise SystemExit(f"the GPU still holds {used} MiB")


def log_tail(path, characters=4000):
    with open(path, errors="replace") as f:
        return f.read()[-characters:]


def wait_for(process, log_path, name, limit_seconds=2400):
    started = reported = time.time()
    while True:
        if process.poll() is not None:
            raise SystemExit(f"{name} exited with code {process.returncode}:\n{log_tail(log_path)}")
        try:
            urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
            print(f"{name} is up after {time.time() - started:.0f} s")
            return
        except Exception:
            pass
        if time.time() - started > limit_seconds:
            raise SystemExit(f"{name} isn't up after {limit_seconds} s:\n{log_tail(log_path)}")
        if time.time() - reported >= 30:
            lines = log_tail(log_path, 1000).strip().splitlines()
            print(f"{time.time() - started:.0f} s, still starting: {lines[-1] if lines else '(no log yet)'}", flush=True)
            reported = time.time()
        time.sleep(2)


def serve(model, extra=""):
    stop_servers()
    log_path = "/content/vllm.log"
    process = subprocess.Popen([f"{VLLM_VENV}/bin/vllm", "serve", model, "--served-model-name", "qwen",
                                "--max-model-len", "16384", "--max-num-seqs", "32", "--gpu-memory-utilization", "0.90",
                                "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
                                "--reasoning-parser", "qwen3", "--port", "8090", *extra.split()],
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    wait_for(process, log_path, "vLLM")


RUN_ARGS = ["--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"]


def check_server():
    os.chdir(EVAL)
    probe = subprocess.run(["python3", "probe_formats.py", "--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen"],
                           capture_output=True, text=True)
    print(probe.stdout, probe.stderr)
    verdicts = dict((line[:32].strip(), line[32:].strip()) for line in probe.stdout.splitlines() if line.strip())
    for required in ("tool round (tool_choice auto)", "full bounded schema"):
        if not verdicts.get(required, "").startswith("accepted"):
            raise SystemExit(f"the server refused the {required}: not a fair run, stop here")

## The teacher, with the harness checked first

On `train-receipts`: `oracle` must grade 1.000 and `silent` 0, and the server must accept the tool round and the full bounded schema. Then come the train runs still to do, and the teacher's evaluation rows if they are missing. Every run resumes from Drive after a disconnect. `train-receipts` holds each train receipt twice, so expect about 40 minutes on an A100 for a dataset of about 1,000 receipts.

In [ ]:
os.chdir(EVAL)
if SERVE_TEACHER:
    serve(TEACHER)
    for name, backend in (("check-oracle", "oracle"), ("check-silent", "silent")):
        graded = subprocess.run(["python3", "run.py", "--suite", TRAIN_SUITE, "--name", name, "--backend", backend,
                                 "--limit", "70", "--restart"], capture_output=True, text=True).stdout
        print(name, [line for line in graded.splitlines() if line.startswith("record_core")])
    check_server()
    for run in NEEDS_TRAIN_RUNS:
        sh(["python3", "run.py", "--suite", TEACHER_RUNS[run]["suite"], "--name", run, *RUN_ARGS])
    for run, suite in NEEDS_EVAL_RUNS:
        sh(["python3", "run.py", "--suite", suite, "--name", run, *RUN_ARGS])
    stop_servers()
else:
    print("the teacher isn't needed" if not NEEDS_EVAL_RUNS else "the teacher's evaluation rows are missing: use an A100")

## The training pairs

`sft_data.py` over every run in `TEACHER_RUNS`, as in `sft_a100.ipynb`. Its counts, written beside the pairs, show how many of the real-receipt sessions the teacher got right (`kept`) and how many come back as doubt pairs.

In [ ]:
!pip install -q -U transformers peft flash-linear-attention
!pip uninstall -q -y torchao
from huggingface_hub import snapshot_download

BASE = "/content/models/qwen35-08b"
snapshot_download(STUDENT, local_dir=BASE)
%cd {EVAL}
if not os.path.exists(PAIRS):
    sh(["python3", "sft_data.py", *TEACHER_RUNS, "--template", BASE, "--tokenizer", BASE,
        "--verify-template", "--max-len", str(MAX_LEN), "--out", PAIRS,
        *(["--doubt-confidence", str(DOUBT_CONFIDENCE)] if DOUBT_CONFIDENCE is not None else [])])
print(open(PAIRS.replace(".jsonl", ".stats.json")).read())

## SFT

A fresh LoRA from the base model on all the pairs, as in `sft_a100.ipynb`. It checkpoints to Drive every 50 steps and resumes when rerun.

In [ ]:
%cd {EVAL}
if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):
    sh(["python3", "train_sft.py", "--model", BASE, "--data", PAIRS,
        "--out", ADAPTER, "--epochs", str(EPOCHS), "--rank", str(RANK), "--lr", str(LR),
        "--accum", str(ACCUM), "--max-len", str(MAX_LEN), "--loss", LOSS, "--save-every", "50"])
!tail -n 3 {ADAPTER}/train_log.jsonl

## Merge

In [ ]:
%cd {EVAL}
if not os.path.exists(f"{MERGED}/merge.json"):
    sh(["python3", "merge_lora.py", "--base", BASE, "--adapter", ADAPTER, "--out", MERGED])
!ls -la {MERGED}

## The merged model on `dev`, `dev-novel` and `dev-receipts`

At bf16 on vLLM. Each suite is compared with the baseline from `real_receipts_eval.ipynb` and the earlier rounds; the first run present is the baseline. On `dev-receipts` that is the shipped file (Q8_0, which costs next to nothing against bf16 on `dev`). `dev-receipts` says whether real receipts got better, and `dev` and `dev-novel` say whether anything else got worse. The audit then shows the cases the new model still gets wrong.

In [ ]:
serve(MERGED, "--dtype bfloat16")
os.chdir(EVAL)
SCORED = {"dev": f"{NAME}-bf16", "dev-novel": f"{NAME}-bf16-dev-novel", EVAL_SUITE: f"{NAME}-bf16-{EVAL_SUITE}"}
for suite, run in SCORED.items():
    keep_on_drive(run)
    sh(["python3", "run.py", "--suite", suite, "--name", run, *RUN_ARGS])
stop_servers()
for run in (f"{SHIPPED}-q8_0-{EVAL_SUITE}", f"qwen3.5-0.8b-fp16-{EVAL_SUITE}"):
    keep_on_drive(run)
LINEUPS = {
    "dev": [f"{EARLIER}-bf16", f"{PREVIOUS}-bf16", f"{SHIPPED}-bf16", SCORED["dev"], TEACHER_NAME],
    "dev-novel": [f"{EARLIER}-bf16-dev-novel", f"{PREVIOUS}-bf16-dev-novel", f"{SHIPPED}-bf16-dev-novel",
                  SCORED["dev-novel"], TEACHER_NOVEL_RUN],
    EVAL_SUITE: [f"{SHIPPED}-q8_0-{EVAL_SUITE}", f"qwen3.5-0.8b-fp16-{EVAL_SUITE}", SCORED[EVAL_SUITE], TEACHER_REAL_RUN],
}
for suite, lineup in LINEUPS.items():
    cases = SUITE_CASES.get(suite, 280)
    present = [run for run in lineup if finished(f"{RECORDS_DIR}/runs/{run}", cases)]
    print(f"\n#### {suite}: {present}")
    if len(present) >= 2:
        sh(["python3", "compare.py", *present])
    for run in present:
        with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
            per_case = json.load(f)["per_case"]
        approved = [c for c in per_case if c["would_approve"]]
        print(f"{run}: approval precision {sum(c['record_all'] for c in approved) / max(1, len(approved)):.2f} "
              f"over {len(approved)} approved")
present = [run for run in (SCORED[EVAL_SUITE], f"{SHIPPED}-q8_0-{EVAL_SUITE}", TEACHER_REAL_RUN)
           if finished(f"{RECORDS_DIR}/runs/{run}", SUITE_CASES[EVAL_SUITE])]
if present:
    sh(["python3", "audit.py", *present, "--limit", "15"])
sh(["python3", "board.py", "--write"])

## Take the results home

As in `sft_a100.ipynb`: for each run, its `meta.json`, `results.jsonl` and `grade.json`, plus the pair counts, the adapter's config and train log, the merge report, the suites' `meta.json` and `RESULTS.md`. The receipts, the suites and the transcripts stay on Drive. Check the dataset's licence before committing the receipt runs: `results.jsonl` holds the drafts, which quote the merchants.

In [ ]:
OUT = "/content/out"
shutil.rmtree(OUT, ignore_errors=True)
for run, folder in TEACHER_DIRS.items():
    os.makedirs(f"{OUT}/runs/{run}")
    for name in ("meta.json", "results.jsonl", "grade.json"):
        shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
os.makedirs(f"{OUT}/sft/{NAME}")
for source in (PAIRS.replace(".jsonl", ".stats.json"), f"{ADAPTER}/adapter_config.json", f"{ADAPTER}/train_args.json",
               f"{ADAPTER}/train_log.jsonl", f"{MERGED}/merge.json"):
    shutil.copy(source, f"{OUT}/sft/{NAME}/")
for run in (TEACHER_NOVEL_RUN, TEACHER_REAL_RUN, *SCORED.values()):
    folder = f"{RECORDS_DIR}/runs/{run}"
    if finished(folder, 1):
        os.makedirs(f"{OUT}/runs/{run}")
        for name in ("meta.json", "results.jsonl", "grade.json"):
            if os.path.exists(f"{folder}/{name}"):
                shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
for name in (TRAIN_SUITE, EVAL_SUITE):
    os.makedirs(f"{OUT}/suites/{name}")
    shutil.copy(f"suites/{name}/meta.json", f"{OUT}/suites/{name}/")
shutil.copy(f"{RECORDS_DIR}/RESULTS.md", OUT)
subprocess.run(f"cd {OUT} && zip -qr /content/{NAME}.zip .", shell=True, check=True)
from google.colab import files
files.download(f"/content/{NAME}.zip")